# Sentence Transformer

????? ???????????? ????????????? ??????, ?? ?????????? sentence-transformers. ?????? ????????, ? ????? ??????, ?? ???? ??????? ???????? ?????? sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2. ????? ???????????? ?? ? ??????? ?????????? sentence-transformers.

In [ ]:
import sys
import os
import warnings

import pandas as pd
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, Features, Value
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import optuna

warnings.filterwarnings('ignore')

# ????????? ? path ??????????? ?????????? ??? ???????  calc_metrics
sys.path.append(os.path.abspath(os.path.join("..")))

from utils import calc_metrics

In [ ]:
metrics = pd.read_csv('metrics.csv', index_col=0)

???????? ??????, ?????? ? ?????? ????? ????????? ????? ?????? ? ??????? ??????????.

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

feature = Features({"release": Value("string")})
dataset = load_dataset("csv", data_files="../data/x.csv", features=feature)
texts = dataset["train"]["release"]
embeddings = model.encode(texts, normalize_embeddings=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"  

In [ ]:
# ???????? ??????
y = pd.read_csv("../data/y.csv", index_col="date").iloc[:, 0]

?????? ?????????? ?????????? ??????? ?? ???? ?????? ????????????? ML. ???????? ??????????????, ? ??????.

In [ ]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 1, 25),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 25),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_features": trial.suggest_int("max_features", 10, 384),
    }

    # ?????????? pipeline
    pipeline = Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    max_depth=params["max_depth"],
                    max_features=params["max_features"],
                    min_samples_leaf=params["min_samples_leaf"],
                    min_samples_split=params["min_samples_split"],
                    n_estimators=params["n_estimators"],
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    # ??????? ??????? ?? ????? ?????????
    metric, _ = calc_metrics(
        embeddings,
        y,
        pipeline,
        name=" ",
        plot=False,
        calc_jobs=-1,
    )

    return metric


# ????????????? ??????? ??????????? optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ????????? ?????? ???????????????
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200, show_progress_bar=True)

????????? ??????? ?????? K-????????? ???????, ???????? ?????????, ???????, ?? ?????? ???????? ??????? ????????? ???.

In [ ]:
study.best_params

?????? ?????? ? ??????? ??????????? ? ????????? ?? ??????????

In [ ]:
pipeline = Pipeline(
    [
        (
            "model",
            RandomForestClassifier(
                max_depth=study.best_params["max_depth"],
                max_features=study.best_params["max_features"],
                min_samples_leaf=study.best_params["min_samples_leaf"],
                min_samples_split=study.best_params["min_samples_split"],
                n_estimators=study.best_params["n_estimators"],
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# ??????? ??????? ?? ????? ?????????
metric, _ = calc_metrics(
    embeddings,
    y,
    pipeline,
    name="Sentence transformer",
    plot=True,
    calc_jobs=-1,
)

In [ ]:
metrics = pd.concat([metrics, metric], axis=0)
metrics

In [ ]:
metrics.to_csv('metrics.csv')